# Artist Marketplace — Search & Ranking Algorithm Design

**Course:** OIT 277 — Digital Platforms in the Age of AI  
**Project:** A Marketplace for Independent Artists  
**Team:** Ru Chen, Smriti Natarajan, Joyce Yu, Hang Yu  
**Companion repo:** `artist-marketplace-algo` worktree (branch: `search-experiments`)

---

## What this notebook is

A **methodology walkthrough** for the search-and-ranking algorithm of our marketplace. It is grounded in the **actual prototype**: same seed data, same schema, same starting ranker. The goal is twofold:

1. **Understand what the prototype does today** — and where it falls short.
2. **Build a comprehensive ranker step by step**, showing what each layer adds, so the procedure is repeatable on future projects.

## What's already in the prototype (commit `4868344`)

The live ranker, in `web/lib/search.ts`, does this:

```
embed(query) → search_works RPC (cosine top-50) → round-robin by artist_id → top-K
```

It works. But it is missing five of the eight things a comprehensive ranker should have. That's the gap this notebook closes.

## The procedure (every stage maps to a class concept)

| Stage | Concept | What we add | Live code? |
|---|---|---|---|
| 0 | The current ranker | (baseline — what we have today) | `lib/search.ts` |
| 1 | The weighted objective `α·rel + β·rev + γ·div + δ·health` | The strategy formula | new |
| 2 | Schema + operational tier definition | How we define "emerging" | reads `artists.reach_score` |
| 3 | Hard filters | Price/medium/license at the SQL level (not client-side) | new (migration) |
| 4 | Semantic retrieval | (already in prototype) | `search_works` RPC |
| 5 | Weighted reranking | MMR + tier boost replaces round-robin | new |
| 6 | Diversity strategies | MMR vs round-robin vs quota | comparison |
| 7 | Cold start + explore/exploit | ε-greedy injection for emerging artists | new |
| 8 | Three-layer evaluation | Offline IR + marketplace + LLM judge | new |
| 9 | Failure mode audit | Negation, ambiguity, zero-match, gaming | new |
| 10 | Agent harness | Confidence tiers, audit log, deployment level | new |

Run the cells top-to-bottom. The embedding step takes ~30 seconds on the full seed (~$0.01). Everything after that is local.

## Notebook ↔ Production divergence (v1)

**This notebook is the design lab — the "ultimate" / aspirational version.**
It includes design alternatives we deliberately did not ship in production v1.

For the live production algorithm, see `algorithm_v1_production.ipynb` and
`web/lib/search.ts`.

| # | This notebook (design) | Production v1 (`web/lib/search.ts`) |
|---|---|---|
| 1 | `gamma_diversity = 0.4` (MMR weight on) | `gamma_diversity = 0.0` — round-robin by artist instead (cheaper, no embeddings server-side) |
| 2 | `RETRIEVAL_K = 30` | `max(K × 6, 60)` ≈ 60 — over-fetch headroom for hard filters + quota |
| 3 | `weighted_rerank()` is greedy MMR — recomputes diversity vs already-shown items | `roundRobinByArtist()` — bucket by `artist_id`, cycle one each |
| 4 | `epsilon_greedy_injection()` — random emerging-artist injection | not in prod (deferred — needs ~1,000+ catalog) |
| 5 | `apply_hard_filters()` returns `drop_log` (audit) | filters return survivors only — no log persisted |
| 6 | `search_agent()` returns full audit log + status codes | rank functions return the array; no audit log persistence |
| 7 | `tier_of(None) → 'emerging'` (benefit of doubt) | `tierOf(reach_score: number)` — assumes seeded value |

**Why the divergence is healthy.** The notebook is the source of truth for
*strategic intent* (weights, floors, philosophy). Production is the
*engineered subset* — predictable, observable, cheap. Each design alternative
above earns its way into production when round-robin / static configuration /
tiny scale stop being adequate. See `notebooks/how_search_works.docx`
Section 6 for the promotion conditions.

— Last sync: 2026-04-25 against `web/lib/search.ts`


## Setup

Only one secret needed: `OPENAI_API_KEY`. The notebook does **not** connect to Supabase — it loads the same `seed_data.json` the prototype's `seed.py` reads, plus the 5 hardcoded artists from that file. Everything happens in-memory.

**In Colab:** add `OPENAI_API_KEY` via the 🔑 sidebar. Then upload `api/scripts/seed_data.json` from the repo (or rely on the inline fallback below).  
**Locally** (notebook lives in `notebooks/`): the cell auto-finds `../api/scripts/seed_data.json`.

In [ ]:
!pip install openai numpy pandas matplotlib --quiet

import os, json, math, time, random
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI

# Colab secret loader, with env-var fallback so the notebook also runs locally
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

assert OPENAI_API_KEY, 'Set OPENAI_API_KEY (Colab secrets or env var).'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBEDDING_MODEL = 'text-embedding-3-small'   # same model the prototype uses

random.seed(42)
np.random.seed(42)

print('✅ Setup complete.')

---
# Stage 0 — The Current Ranker (Baseline)

Before designing anything new, we need to **understand what the prototype does today** and run it on real data so we have a baseline to compare against. This is the most important habit when redesigning a system: **measure the thing you're replacing.**

## What `lib/search.ts` does, in plain English

1. **`embed(query)`** — call OpenAI `text-embedding-3-small`, get a 1536-dim vector.
2. **`search_works` RPC** — Postgres returns the top-50 works ranked by cosine similarity (`1 - (embedding <=> query_embedding)`). HNSW index, no filters.
3. **`rankWorks(...)`** in TypeScript — bucket by `artist_id`, then **round-robin**: take the best work from each artist, then a second from each, etc., until we hit `limit`.

## What it does NOT do

| Missing | Consequence |
|---|---|
| Hard filters at the SQL level (price, medium, license) | Frontend filters happen *after* a fixed top-50 — you can get "0 results" even when matches exist |
| Emerging-artist boost | Every artist gets one slot, but a high-reach veteran with one match beats a low-reach newcomer with five matches |
| Relevance floor | A query with no good match still returns the highest-of-bad with full confidence |
| Score breakdown / audit log | Result rows have one `score` field — no way to explain "why" |
| Cold-start handling | New artists with no interaction data are invisible to bandit-style mechanisms |

Below: load the real seed and reproduce the current ranker exactly.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load the prototype's real seed data.
#
#   - 5 hardcoded artists from api/scripts/seed.py (the original design)
#   - Synthetic artists from api/scripts/seed_data.json (LLM-generated)
#
# If the JSON isn't reachable (e.g. fresh Colab without upload), we fall
# back to a curated 12-artist subset hardcoded below. That keeps the
# notebook self-runnable while preferring real data when available.
# ═══════════════════════════════════════════════════════════════

ARTISTS_HARDCODED = [
    {'display_name':'Ava Chen',  'bio':'Minimalist illustrator, pastel palettes, coastal themes.',
     'location':'Toronto, CA',  'attestation_tier':'verified',     'reach_score':6400,
     'works':[{'title':'Ocean fog','description':'soft watercolor of coastal fog at dawn, muted blues and greys','medium':'illustration','price_from_cents':14400},
              {'title':'City rain','description':'neon-tinged illustration of rain on asphalt at night','medium':'illustration','price_from_cents':12000}]},
    {'display_name':'Noah Patel','bio':'Indie lofi beatmaker blending rain samples and tape hiss.',
     'location':'Brooklyn, NY','attestation_tier':'verified',     'reach_score':12000,
     'works':[{'title':'Midnight loop','description':'lofi hip-hop instrumental with soft rain and muted drums','medium':'music','price_from_cents':6500},
              {'title':'Window light','description':'ambient piano piece with warm tape saturation','medium':'music','price_from_cents':7200}]},
    {'display_name':'Maya Gomez','bio':'Character designer for indie games.',
     'location':'CDMX, MX',    'attestation_tier':'self_declared','reach_score':3800,
     'works':[{'title':'Fox warrior','description':'armored kitsune warrior character concept in ink and watercolor','medium':'character','price_from_cents':14000},
              {'title':'Sky whale','description':'gentle floating sky whale creature with trailing seaweed','medium':'character','price_from_cents':16500}]},
    {'display_name':'Leo Brown','bio':'Documentary cinematographer focused on quiet landscapes.',
     'location':'Portland, OR','attestation_tier':'self_declared','reach_score':1100,
     'works':[{'title':'Quiet harbor','description':'short video study of a fog-shrouded harbor at first light','medium':'video','price_from_cents':10500},
              {'title':'Tidepool','description':'macro video of a tidepool, slow camera drift over kelp','medium':'video','price_from_cents':9000}]},
    {'display_name':'Sora Kim','bio':'Multidisciplinary artist, synth soundscapes and abstract visuals.',
     'location':'Berlin, DE',  'attestation_tier':'self_declared','reach_score':2300,
     'works':[{'title':'Aurora drift','description':'ambient synth drone evoking slow polar light','medium':'music','price_from_cents':4500}]},
]

INLINE_FALLBACK = [   # 12-artist representative subset (used only if JSON not found)
    {'display_name':'Kenji Arai','bio':'Long-form drones and environmental recordings, urban Kyoto.',
     'location':'Kyoto, JP','attestation_tier':'self_declared','reach_score':8000,
     'works':[{'title':'Whispers of Rain','description':'Layered tape loops capture the delicate sound of rainfall, woven with soft synth drones.','medium':'music','price_from_cents':4000},
              {'title':'Dawn Chorus','description':'Field recordings of chirping birds merge with slow-moving textures, immersive early morning.','medium':'music','price_from_cents':4000}]},
    {'display_name':'Marcus Winter','bio':'Cinematic ambient and orchestral hybrid composer.',
     'location':'Reykjavik, IS','attestation_tier':'verified','reach_score':9200,
     'works':[{'title':'Glacial Pulse','description':'Slow-building orchestral piece with cinematic strings and deep subbass for trailer cold opens.','medium':'music','price_from_cents':12000},
              {'title':'Northern Quiet','description':'Sparse piano with warm tape saturation, suited to documentary openings.','medium':'music','price_from_cents':12000}]},
    {'display_name':'Aya Tolentino','bio':'Bedroom-pop producer, tape-warm beats.',
     'location':'Manila, PH','attestation_tier':'self_declared','reach_score':900,
     'works':[{'title':'Slow Window','description':'Lo-fi instrumental with vinyl crackle and chopped Rhodes, suited to evening vlog content.','medium':'music','price_from_cents':3500}]},
    {'display_name':'Pilar Vega','bio':'Editorial illustrator, bold flat colors.',
     'location':'Madrid, ES','attestation_tier':'verified','reach_score':11000,
     'works':[{'title':'Plaza at Dusk','description':'Bold flat-color illustration of a Madrid plaza at dusk, warm palette, editorial style.','medium':'illustration','price_from_cents':16000},
              {'title':'Sunday Market','description':'Flat-color illustration of a weekly market, vibrant, suitable for magazine spreads.','medium':'illustration','price_from_cents':16000}]},
    {'display_name':'Mireia Pons','bio':'Hand-drawn line art, minimalist editorial.',
     'location':'Barcelona, ES','attestation_tier':'self_declared','reach_score':5200,
     'works':[{'title':'Botanical I','description':'Hand-drawn black-and-white botanical study, clean and editorial.','medium':'illustration','price_from_cents':6000}]},
    {'display_name':'Juno Fernández','bio':'Surrealist digital painter.',
     'location':'Buenos Aires, AR','attestation_tier':'self_declared','reach_score':700,
     'works':[{'title':'Dream Tide','description':'Surreal digital painting of a flooded city at dusk, dreamlike palette.','medium':'illustration','price_from_cents':4000}]},
    {'display_name':'Ines Moreau','bio':'Documentary cinematographer, slow nature studies.',
     'location':'Lyon, FR','attestation_tier':'verified','reach_score':7800,
     'works':[{'title':'Loire Light','description':'Slow cinematic pans of mist over the Loire river, suited to luxury travel content.','medium':'video','price_from_cents':18000},
              {'title':'Forest Hours','description':'Long-take footage of beech forests in autumn, suitable for AI training datasets.','medium':'video','price_from_cents':18000}]},
    {'display_name':'Raza Kahn','bio':'Street documentary filmmaker.',
     'location':'Karachi, PK','attestation_tier':'self_declared','reach_score':3600,
     'works':[{'title':'Saddar Nights','description':'Documentary-style street footage of Karachi at dusk, neon reflections, vlog-friendly.','medium':'video','price_from_cents':12000}]},
    {'display_name':'Oscar Dias','bio':'Drone operator and aerial videographer.',
     'location':'Lisbon, PT','attestation_tier':'self_declared','reach_score':400,
     'works':[{'title':'Coastal Drift','description':'4K drone footage of Atlantic coastline cliffs, slow cinematic pans.','medium':'video','price_from_cents':8000}]},
    {'display_name':'Asher Coen','bio':'Original character designer, indie game IP.',
     'location':'Tel Aviv, IL','attestation_tier':'self_declared','reach_score':6100,
     'works':[{'title':'Forest Mascots','description':'Original character set of friendly forest mascots in flat illustrative style, suited to onboarding apps.','medium':'character','price_from_cents':16500}]},
    {'display_name':'Wren Hollis','bio':'Mascot illustrator for fintech and SaaS.',
     'location':'Austin, TX','attestation_tier':'self_declared','reach_score':1100,
     'works':[{'title':'Coin Critters','description':'Original quirky animal mascots designed for fintech onboarding, friendly and approachable.','medium':'character','price_from_cents':9000}]},
    {'display_name':'Kenneth Eko','bio':'Afrofuturist character designer.',
     'location':'Lagos, NG','attestation_tier':'self_declared','reach_score':300,
     'works':[{'title':'Solar Voyager','description':'Original character concept of an Afrofuturist astronaut in cel-shaded style.','medium':'character','price_from_cents':6000}]},
]

def load_synthetic_subset():
    candidate_paths = [
        Path('../api/scripts/seed_data.json'),                        # local repo run
        Path('/Users/joyce/Desktop/artist_marketplace/api/scripts/seed_data.json'),  # absolute fallback
        Path('seed_data.json'),                                       # Colab upload
    ]
    for p in candidate_paths:
        if p.exists():
            data = json.loads(p.read_text())
            picks = {'Kenji Arai','Marcus Winter','Aya Tolentino','Pilar Vega','Mireia Pons',
                     'Juno Fernández','Ines Moreau','Raza Kahn','Oscar Dias','Asher Coen',
                     'Wren Hollis','Kenneth Eko'}
            sub = [a for a in data if a['display_name'] in picks]
            for a in sub:
                a['works'] = a['works'][:2]   # cap at 2 works/artist for notebook size
            print('Loaded {} artists from {}'.format(len(sub), p))
            return sub
    print('seed_data.json not found — using inline fallback ({} artists)'.format(len(INLINE_FALLBACK)))
    return INLINE_FALLBACK

ALL_ARTISTS = ARTISTS_HARDCODED + load_synthetic_subset()

# Flatten into a works-level catalog (one row per work) — same shape the SQL `works` table has
rows = []
for a in ALL_ARTISTS:
    for w in a['works']:
        rows.append({
            'work_id':         '{}/{}'.format(a['display_name'], w['title']),
            'artist':          a['display_name'],
            'bio':             a['bio'],
            'location':        a['location'],
            'attestation_tier':a['attestation_tier'],
            'reach_score':     a['reach_score'],
            'title':           w['title'],
            'description':     w['description'],
            'medium':          w['medium'],
            'price_from_cents':w['price_from_cents'],
        })
catalog = pd.DataFrame(rows)
print('\n✅ Catalog: {} works · {} artists · {} mediums'.format(
    len(catalog), catalog['artist'].nunique(), catalog['medium'].nunique()))
print('   tiers: ', dict(catalog.groupby('attestation_tier').size()))
print('   mediums:', dict(catalog.groupby('medium').size()))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Embed the catalog (one-time, ~30s for ~50 works, costs <$0.01).
# Same model and same input formula as the prototype's seed.py:
#     embedding_input = f"{title}. {description}"
# ═══════════════════════════════════════════════════════════════

def embed_text(text):
    if not text or not text.strip():
        return None
    r = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=text.strip())
    return np.array(r.data[0].embedding, dtype=np.float32)

print('🧠 Embedding {} works...'.format(len(catalog)))
embeddings = []
for _, row in catalog.iterrows():
    embeddings.append(embed_text(f"{row['title']}. {row['description']}"))
    time.sleep(0.02)
EMB = np.vstack(embeddings)
EMB_NORM = EMB / np.linalg.norm(EMB, axis=1, keepdims=True)   # pre-normalize → cosine = dot product
print('✅ Done. Each vector: {} dims.'.format(EMB.shape[1]))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# THE CURRENT RANKER — exact reproduction of lib/search.ts
# ═══════════════════════════════════════════════════════════════
#
# This is the algorithm running in production right now. We reproduce
# it 1:1 in Python so we can run it on the same data as our proposed
# replacements. Every line below has a counterpart in lib/search.ts.
# ═══════════════════════════════════════════════════════════════

def search_works_rpc(query_vec, match_count=50):
    """Mirrors the Postgres `search_works` RPC: cosine top-N over the whole catalog."""
    q = query_vec / np.linalg.norm(query_vec)
    sims = EMB_NORM @ q
    top_idx = np.argsort(-sims)[:match_count]
    return [{'idx': int(i), 'similarity': float(sims[i]),
             **catalog.iloc[int(i)].to_dict()} for i in top_idx]

def rank_works_current(query, limit=10):
    """Reproduces lib/search.ts `rankWorks`: round-robin by artist_id over a 5x over-fetch."""
    q_vec = embed_text(query)
    rows = search_works_rpc(q_vec, match_count=limit * 5)

    by_artist = defaultdict(list)
    order = []
    for r in rows:
        if r['artist'] not in by_artist:
            order.append(r['artist'])
        by_artist[r['artist']].append(r)

    merged = []
    while len(merged) < limit and any(by_artist[a] for a in order):
        for a in order:
            if by_artist[a]:
                merged.append(by_artist[a].pop(0))
                if len(merged) >= limit:
                    break
    return merged

# Run the current ranker on a real query — exactly like a user would on /search
demo_query = 'moody lofi music for a quiet vlog'
print('Query:', demo_query)
print('Ranker: cosine + round-robin by artist_id  (production behavior)\n')
for i, r in enumerate(rank_works_current(demo_query, limit=8), 1):
    print('  {:>2}. {:.2f}  | {:14s} ({:13s} reach={:>5})  | {:6s} | {}'.format(
        i, r['similarity'], r['artist'][:14], r['attestation_tier'],
        r['reach_score'], r['medium'], r['title']))

### What you should see in the baseline

Look at the output above. Two patterns are likely visible already:

1. **Mediums are mixed.** A query about "music" returned illustrations and videos because there's no medium filter. The frontend would filter those out *after* the response, possibly leaving the user with very few results.
2. **High-reach artists tend to crowd the top** — round-robin guarantees one slot each, but it doesn't reward the long tail. An emerging artist (`reach_score < 1500`) with a strong match is treated identically to a celebrity with a weaker one.

These are the two specific gaps Stages 1–5 will close. Stages 6–10 add evaluation and the agent harness on top.

---

# Stage 1 — Frame the Objective

The single most consequential decision in any ranker is **what you're optimizing for**. Session 4 gave us the canonical formulation; every ranker on the planet is some version of this:

$$\text{score}(i) = \alpha \cdot \text{rel}(i) + \beta \cdot \text{rev}(i) + \gamma \cdot \text{div}(i, \text{shown}) + \delta \cdot \text{health}(i) - \varepsilon \cdot \text{price\_penalty}(i, B)$$

**The weights are the platform's strategy.** Today the prototype effectively has `α=1.0, γ=very_small (round-robin only), β=δ=ε=0`.

## Our strategic stance: **emerging-artist exposure is the lead metric**

This is *not* the default move for a search system — most platforms lead with relevance and bolt diversity / fairness on top. Our project proposal §2 commits to the opposite stance: *"the platform collapses into a winner-take-all feed dominated by the same few names"* without the long-tail boost. So:

> **Relevance is in service of emerging-artist exposure**, not the other way around.

That's why δ is meaningful (not a tiny add-on), and why we also add a **quota** on top of the soft weight (Stage 6) — belt and braces.

## What each term means *for our marketplace*

| Term | Meaning | Operationalization |
|---|---|---|
| **α — Relevance** | How well does this work match the buyer's brief? | Cosine similarity (already in prototype) |
| **β — Revenue** | Expected platform earnings from surfacing this | `price_from_cents × past_conversion_rate × take_rate` (we don't have conversion data yet → β=0 in v1) |
| **γ — Diversity** | Does the result set span styles/artists? | MMR penalty: how different is this work from already-shown ones (across all artists, not just by `artist_id`) |
| **δ — Marketplace health** | Does the long tail get a chance? | Bonus for emerging-tier artists who clear a relevance floor |
| **ε — Price penalty** *(new)* | Soft constraint on going over budget | Quadratic ramp from 0 (within budget) to ε at 1.5× budget; hard-rejected beyond |

## Why we're not turning β on yet

Session 4's **Etsy failure mode**: when ranking optimizes for volume/conversion, drop-shippers win and handmade sellers get squeezed out. *"The ranking objective became a de facto marketplace policy."* We don't have conversion data yet — and even when we do, β should ramp slowly. Our entire value prop is the long tail.

## v1 weights (locked via group discussion)

Below: relevance-led, **moderate δ** (not a token weight), MMR diversity, soft price penalty active. β stays off.


In [ ]:
# NOTE — divergences from production v1 in this cell:
#   • gamma_diversity = 0.0 in prod (round-robin replaces MMR)
#   • RETRIEVAL_K = max(K × 6, 60) ≈ 60 in prod (this 30 is
#     the design-lab value; prod needs headroom because hard filters
#     run AFTER retrieval, not before)
# γ=0.4 stays here as the v2 MMR design candidate.

# NOTE — production v1 over-fetches `max(K × 6, 60) = 60` candidates
# by default. The 30 here is the design-lab value; production needs
# more headroom because hard filters run after retrieval, not before.

# NOTE — production v1 ships `gamma_diversity = 0.0`. Round-robin by
# artist handles diversity mechanically (cheaper than MMR, no embeddings
# fetched server-side). γ=0.4 here is the v2 MMR design candidate.

# ═══════════════════════════════════════════════════════════════
# THE WEIGHTED OBJECTIVE — single config the product team owns.
# Everything downstream reads from this dict. Strategy changes
# happen here, not in algorithm code.
# ═══════════════════════════════════════════════════════════════

WEIGHTS = {
    'alpha_relevance':  1.0,   # base: cosine match (always on)
    'beta_revenue':     0.0,   # off in v1 — see Etsy failure mode
    'gamma_diversity':  0.4,   # MMR-style penalty across the whole catalog
    'delta_health':     0.5,   # MODERATE boost for emerging artists (was 0.3 — bumped per group strategy)
}

# New configs introduced in this version of the notebook:

EPSILON_PRICE_PENALTY = 0.6   # max contribution of price penalty to the score
                              # (subtracted, so this is the maximum points you LOSE
                              # for being just under the 1.5× ceiling)

PRICE_CEILING_MULT    = 1.5   # hard-reject works priced above 1.5 × buyer's budget;
                              # within [budget, ceiling] is the SOFT band

QUOTA_FLOOR_REL       = 0.40  # quota slots only go to emerging works that cleared
                              # at least this much relevance — quota is permissive,
                              # not coercive

RETRIEVAL_K           = 30    # how many candidates the embedding stage returns
FINAL_K               = 8     # how many results the buyer sees
MIN_RELEVANCE_FLOOR   = 0.20  # below this, δ-boost cannot rescue a weak match

print('Weights:', WEIGHTS)
print('Price soft band: penalty up to {:.2f} at {:.1f}× budget; hard reject beyond'.format(
    EPSILON_PRICE_PENALTY, PRICE_CEILING_MULT))
print('Quota relevance floor:', QUOTA_FLOOR_REL)


---
# Stage 2 — Operational Tier Definition

The δ term boosts "emerging" artists. But the prototype schema has **two** signals related to artists' standing:

| Column | Meaning | Type |
|---|---|---|
| `artists.attestation_tier` | identity verification (`verified` / `self_declared`) | binary |
| `artists.reach_score` | follower count (cached) | continuous |

**These are different axes.** `attestation_tier` says *"is this person who they claim to be?"* (identity). `reach_score` says *"how much exposure does this person already have?"* (market access).

The δ term is about **fixing the discovery problem**, which is about *exposure*, not identity. So:

**Operational definition we'll use:**

| Tier | Condition | δ-bonus |
|---|---|---|
| `emerging`   | `reach_score < 1500`  | +1.0 |
| `growing`    | `1500 ≤ reach_score < 5000` | +0.5 |
| `established`| `reach_score ≥ 5000`  | 0.0 |

These boundaries are intentionally the same as the multipliers in `seed.py`'s `_price_for()`, so the same buckets we use for pricing align with the buckets we use for discovery. **No penalty for established artists** — we just give the long tail a lift, not a punishment.

## Discussion question

Should `attestation_tier = verified` *also* feed something? E.g., a separate **trust** term? It's not the same as exposure — a verified emerging artist is more trustworthy than an unverified one but still needs the exposure boost. Open question for the group.

In [ ]:
TIER_BONUS = {'emerging': 1.0, 'growing': 0.5, 'established': 0.0}

def tier_of(reach_score):
    if reach_score is None:           return 'emerging'   # no signal → benefit of the doubt
    if reach_score < 1500:            return 'emerging'
    if reach_score < 5000:            return 'growing'
    return 'established'

catalog['tier'] = catalog['reach_score'].apply(tier_of)

print('Tier distribution in our catalog:')
print(catalog.groupby(['tier','attestation_tier']).size().unstack(fill_value=0))

---
# Stage 3 — Hard Filters and the Price Soft Band

Today the prototype handles filters in two wrong places:

1. **Refine chips** (`under $200`, `this week`, `warmer`) → appended as text to the query, then embedded.
   Bug: the model treats *"under $200"* as a vibe, not a constraint. A $1,500 work whose description vaguely matches will still rank.

2. **Medium / verified-only** → filtered in `SearchPage.tsx` *after* the API responds.
   Bug: if the top-50 retrieved have only 2 music works, but the DB has 50 more, the user never sees them.

## Hard vs. soft — the v1 split (locked via group discussion)

| Constraint | Type | Why |
|---|---|---|
| **Medium** (music / illustration / video / character) | HARD | A buyer asking for music doesn't want video at any score |
| **Verified-only** (when buyer toggles it) | HARD | Identity guarantee is a yes/no |
| **Price** | **SOFT within a band, HARD beyond** | Buyer says "under $200" — but if a $250 work is exceptional, surface it. A $1,000 work, never |
| **Refine chips** (`warmer`, `darker`, `more tape`) | SOFT (stay in embedded query) | These are vibes — perfect for embeddings to handle |

## How the price soft band works

A buyer sets budget `B`. Then:

- `price ≤ B` → no penalty (`price_pen = 0`)
- `B < price ≤ 1.5·B` → quadratic penalty, ramping to ε at the ceiling. *Small overruns barely hurt; large ones hurt a lot.*
- `price > 1.5·B` → **hard reject** — never surfaced

This means: a $220 work against a $200 budget loses a tiny fraction of its score, and an exceptional match easily survives. A $290 work loses a substantial chunk and only wins if relevance is exceptional. A $310 work is gone.

## What this looks like as a SQL change (Appendix shows the migration)

The Postgres RPC will need to take `budget_cents` as a parameter and apply the ceiling check (`price ≤ budget × 1.5`) in the `WHERE` clause. The soft penalty itself is computed in TypeScript / Python, not in SQL.

Below: the Python implementation, mirroring what the production code will do.


In [ ]:
# NOTE — production v1's `applyHardFilters()` returns survivors only;
# the `drop_log` audit trail here is deferred (see how_search_works.docx §6).

def price_soft_penalty(price_cents, budget_cents,
                       ceiling_mult=PRICE_CEILING_MULT):
    """Return a normalized penalty in [0, 1] for prices in the soft band.

    - 0       when price ≤ budget         (no penalty, full score)
    - quadratic ramp 0 → 1 across [budget, budget × ceiling_mult]
    - 1       at or beyond the ceiling    (caller hard-rejects)

    The actual subtraction in the score is `epsilon_price * price_soft_penalty(...)`,
    so this function returns just the curve. The quadratic shape means small
    overruns ($210 vs $200) barely matter, but large ones ($290 vs $200) really
    sting — exactly what the strategy calls for.
    """
    if budget_cents is None or price_cents <= budget_cents:
        return 0.0
    ceiling = budget_cents * ceiling_mult
    if price_cents >= ceiling:
        return 1.0
    over = (price_cents - budget_cents) / (ceiling - budget_cents)   # ∈ [0, 1]
    return over * over


def apply_hard_filters(df, *, budget_cents=None, price_min_cents=None,
                       medium=None, verified_only=False,
                       ceiling_mult=PRICE_CEILING_MULT):
    """Drop rows that violate any HARD requirement. Returns (filtered, drop_log).

    * `budget_cents`    triggers a HARD reject of anything > budget × ceiling_mult.
                        Items in the soft band [budget, ceiling] survive — they
                        get penalized later by `price_soft_penalty` in the ranker.
    * `medium`          hard filter (string or list of strings)
    * `verified_only`   when True, drops self-declared artists
    * `price_min_cents` strict lower bound (rare — "premium only" buyer)

    The `drop_log` records WHY each row was dropped — that's the audit trail
    the UI uses to say "X was filtered because Y".
    """
    out = df.copy()
    drop_log = []

    if budget_cents is not None:
        ceiling = int(budget_cents * ceiling_mult)
        viol = out[out['price_from_cents'] > ceiling]
        for _, r in viol.iterrows():
            drop_log.append((r['work_id'], 'price_ceiling',
                            '${} > ${} ({}× $-{} budget)'.format(
                                r['price_from_cents']//100, ceiling//100,
                                ceiling_mult, budget_cents//100)))
        out = out[out['price_from_cents'] <= ceiling]

    if price_min_cents is not None:
        viol = out[out['price_from_cents'] < price_min_cents]
        for _, r in viol.iterrows():
            drop_log.append((r['work_id'], 'price_min',
                            '${} < ${}'.format(r['price_from_cents']//100, price_min_cents//100)))
        out = out[out['price_from_cents'] >= price_min_cents]

    if medium is not None:
        wanted = medium if isinstance(medium, list) else [medium]
        viol = out[~out['medium'].isin(wanted)]
        for _, r in viol.iterrows():
            drop_log.append((r['work_id'], 'medium', '{} not in {}'.format(r['medium'], wanted)))
        out = out[out['medium'].isin(wanted)]

    if verified_only:
        viol = out[out['attestation_tier'] != 'verified']
        for _, r in viol.iterrows():
            drop_log.append((r['work_id'], 'verified_only', r['attestation_tier']))
        out = out[out['attestation_tier'] == 'verified']

    return out.reset_index(drop=True), drop_log


# Demo: budget $80 → ceiling $120, music only
demo, log = apply_hard_filters(catalog, budget_cents=8000, medium='music')
print('Music works within $0–$120 (1.5× $80 budget): {} survive'.format(len(demo)))
print('Sample drops (note "price_ceiling" = strictly over $120 hard cap):')
for wid, reason, detail in log[:5]:
    print('  - {:35s} ({}: {})'.format(wid, reason, detail))

# Show the soft-band penalty curve so you can see the strategy in action
print('\nSoft-band penalty curve (budget=$80, ceiling=$120):')
for p in [60, 80, 90, 100, 110, 120, 130]:
    pen = price_soft_penalty(p*100, 8000)
    bar = '█' * int(pen * 30) if pen > 0 else ''
    note = '  ← hard rejected by filter' if p > 120 else ''
    print('  ${:>3}  →  curve {:.2f}  {}{}'.format(p, pen, bar, note))


---
# Stage 4 — Semantic Retrieval (already in prototype)

This stage doesn't change. The prototype already does it correctly:

- Embed the query with the same model used for the catalog
- Cosine similarity against the pre-indexed work embeddings (HNSW index)
- Take the top-K

The only design choice: **what context to embed**. The prototype uses `f"{title}. {description}"`. Session 4 warned: *"Title only vs. title + description + reviews. More context = richer vectors, can blur signal."* We'll keep `title + description` for now and revisit when we have user clicks to fine-tune on (Airbnb-style — Session 4).

In [ ]:
def semantic_retrieve(query, candidate_df, candidate_emb_norm, top_k=RETRIEVAL_K):
    """Return top_k items from candidate_df ranked by cosine similarity to the query.

    candidate_emb_norm: pre-normalized embedding rows aligned with candidate_df.
    """
    q = embed_text(query)
    q = q / np.linalg.norm(q)
    sims = candidate_emb_norm @ q
    out = candidate_df.copy()
    out['relevance'] = sims
    return out.sort_values('relevance', ascending=False).head(top_k).reset_index(drop=True)

def emb_for(df_subset):
    """Aligned embedding matrix for any subset of `catalog` (preserves order of df_subset)."""
    idx = [catalog.index[catalog['work_id']==w].tolist()[0] for w in df_subset['work_id']]
    return EMB_NORM[idx]

---
# Stage 5 — Weighted Reranking (the real change)

Now we apply the full weighted objective on top of the relevance score, replacing the round-robin step.

$$\text{final}(i) = \alpha \cdot \text{rel}(i) + \beta \cdot \text{rev}(i) + \gamma \cdot \text{div}(i, \text{shown}) + \delta \cdot \text{health}(i) - \varepsilon \cdot \text{price\_penalty}(i, B)$$

## How each component is computed

- **`rel`** — cosine similarity from Stage 4.
- **`rev`** — `(price_from_cents × past_conversion_rate)`, min-max normalized to [0, 1]. Currently always 0 because we don't yet track conversions. The column exists so β can turn on without code changes once we have data.
- **`div`** — **Maximal Marginal Relevance.** For each candidate during selection, compute `1 - max(cosine to anything already selected)`. Range [0, 1]. The first item shown gets div=1.0 (nothing to compare to).
- **`health`** — `TIER_BONUS[tier_of(reach_score)]`. Only applied if `rel ≥ MIN_RELEVANCE_FLOOR` so δ can't rescue irrelevant matches.
- **`price_pen`** *(new)* — `price_soft_penalty(price, budget)` from Stage 3. Subtracted from the score; weight is `EPSILON_PRICE_PENALTY` (default 0.6). Only nonzero for over-budget works in the soft band.

## Why MMR instead of round-robin?

Round-robin (current) only knows about `artist_id`. It can't notice that **two different artists are stylistically nearly identical** — both lo-fi music, similar tempo, similar mood. MMR penalizes embedding-similarity to already-selected items, so it diversifies along *style*, not just artist identity. That's the difference between a marketplace that exposes *range* vs. one that exposes *names*.

## Why a relevance floor on δ?

Without it, an emerging artist with `rel=0.10` would beat an established artist with `rel=0.85` once you turn δ up. That's the algorithm gaming itself. The floor (default 0.20) says *"δ only matters among items that already cleared a basic relevance bar."*

## Auditability — every result row carries its breakdown

Project proposal §5 commits to *"log the semantic similarity and diversity adjustment separately."* Implemented via the `breakdowns` list below — every result has `rel`, `rev`, `div`, `health`, `price_pen`, and `final_score`, ready to be returned in the API response and shown in the UI.


In [ ]:
# NOTE — this greedy MMR reranker is NOT what production v1 runs.
# Production uses `roundRobinByArtist()` (see algorithm_v1_production.ipynb).
# Kept here as the v2 candidate when round-robin produces visibly
# clustered results in real traffic.

def revenue_proxy(df):
    """Min-max normalized expected value. Returns zeros until conversion data exists."""
    if 'past_conversion_rate' not in df.columns:
        return np.zeros(len(df))
    raw = df['price_from_cents'].values * df['past_conversion_rate'].values
    if raw.max() == raw.min():
        return np.zeros_like(raw, dtype=float)
    return (raw - raw.min()) / (raw.max() - raw.min())

def health_score(df):
    return df['tier'].map(TIER_BONUS).fillna(0.0).values

def diversity_score(candidate_emb_norm_row, shown_emb_norm_matrix):
    """1 - max cosine to anything already shown. First item: 1.0 (nothing to compare)."""
    if shown_emb_norm_matrix is None or len(shown_emb_norm_matrix) == 0:
        return 1.0
    sims = shown_emb_norm_matrix @ candidate_emb_norm_row
    return float(1.0 - sims.max())

def weighted_rerank(retrieved_df, retrieved_emb_norm, weights, final_k=FINAL_K,
                   min_rel=MIN_RELEVANCE_FLOOR, budget_cents=None,
                   epsilon_price=EPSILON_PRICE_PENALTY):
    """Greedy weighted reranker with full score breakdown per result.

    Greedy because γ depends on what's already chosen. Each iteration:
    score every remaining candidate, pick the best, append, repeat.

    score = α·rel + β·rev + γ·div + δ·health − ε·price_penalty

    `budget_cents` activates the price soft penalty. If None, price_pen=0.
    """
    df = retrieved_df.copy().reset_index(drop=True)
    df['revenue_norm'] = revenue_proxy(df)
    df['health_norm']  = health_score(df)

    selected, breakdowns = [], []
    remaining = set(df.index.tolist())

    while remaining and len(selected) < final_k:
        shown_emb = retrieved_emb_norm[selected] if selected else None
        best_idx, best_score, best_components = None, -1e9, None
        for i in remaining:
            rel = float(df.loc[i, 'relevance'])
            health_eff = float(df.loc[i, 'health_norm']) if rel >= min_rel else 0.0
            div = diversity_score(retrieved_emb_norm[i], shown_emb)
            rev = float(df.loc[i, 'revenue_norm'])
            price_pen = (price_soft_penalty(int(df.loc[i, 'price_from_cents']), budget_cents)
                         if budget_cents is not None else 0.0)
            score = (weights['alpha_relevance']  * rel
                     + weights['beta_revenue']    * rev
                     + weights['gamma_diversity'] * div
                     + weights['delta_health']    * health_eff
                     - epsilon_price              * price_pen)
            if score > best_score:
                best_score, best_idx = score, i
                best_components = {'rel': rel, 'rev': rev, 'div': div,
                                   'health': health_eff, 'price_pen': price_pen,
                                   'final_score': score}
        selected.append(best_idx)
        remaining.remove(best_idx)
        breakdowns.append({**df.loc[best_idx].to_dict(), **best_components,
                          'rank': len(selected)})

    return pd.DataFrame(breakdowns)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Side-by-side: CURRENT (round-robin) vs PROPOSED (weighted) on
# the same query, same hard filters. This is the artifact for the
# group meeting — "see what each layer adds."
# ═══════════════════════════════════════════════════════════════

demo_query = 'moody lofi music for a quiet vlog'
demo_filters = {'medium': 'music', 'budget_cents': 6500}   # $65 budget, $97.50 ceiling

# --- CURRENT path (no SQL filters; client-side filter after) ---
current = rank_works_current(demo_query, limit=8)
current_filtered = [r for r in current if r['medium']=='music' and r['price_from_cents']<=6500]

# --- PROPOSED path: filter → retrieve → weighted rerank (with price penalty) ---
filt, _ = apply_hard_filters(catalog, **demo_filters)
retr = semantic_retrieve(demo_query, filt, emb_for(filt))
proposed = weighted_rerank(retr, emb_for(retr), WEIGHTS, final_k=8,
                           budget_cents=demo_filters['budget_cents'])

print('Query: "{}"   Filters: {}'.format(demo_query, demo_filters))
print('  → budget=$65, hard ceiling=$97 (1.5× budget); above $65 gets soft penalty\n')

print('=== CURRENT ranker (cosine + round-robin, then client-side filter) ===')
for i, r in enumerate(current_filtered, 1):
    print('  {:>2}. rel={:.2f} | {:14s} ({:11s}) | ${:>3} | {}'.format(
        i, r['similarity'], r['artist'][:14], tier_of(r['reach_score']),
        r['price_from_cents']//100, r['title']))
print('  → returned {} results to user (after client-side filter)\n'.format(len(current_filtered)))

print('=== PROPOSED ranker (filter → weighted rerank with ε·price_pen) ===')
for _, r in proposed.iterrows():
    over_budget = '*' if r['price_from_cents'] > demo_filters['budget_cents'] else ' '
    print('  {:>2}. final={:.2f} (rel={:.2f} div={:.2f} health={:.2f} pen={:.2f}) | {:14s} ({:11s}) | ${:>3}{} | {}'.format(
        int(r['rank']), r['final_score'], r['rel'], r['div'], r['health'], r['price_pen'],
        r['artist'][:14], r['tier'], r['price_from_cents']//100, over_budget, r['title']))
print('   (* = over budget but within soft band)')


### Read the breakdown

The proposed ranker exposes WHY each item ranked: `rel` (semantic match), `div` (1 − max-similarity-to-shown), `health` (emerging boost), and `final_score`. When a teammate asks *"why this order?"*, point at this table.

**This is the audit trail proposal §4 commits to** ("transparent scoring for the prototype so we can inspect and evaluate the system").

In [ ]:
# 🎛️ Weight sweep — same query, four strategies. Look at how the artist
# mix and tier mix shift across strategies.

strategies = {
    'pure_relevance':     {'alpha_relevance':1.0,'beta_revenue':0.0,'gamma_diversity':0.0,'delta_health':0.0},
    'rel_plus_diversity': {'alpha_relevance':1.0,'beta_revenue':0.0,'gamma_diversity':0.5,'delta_health':0.0},
    'our_proposal':       WEIGHTS,         # δ=0.5 — emerging-first
    'tier_lean':          {'alpha_relevance':1.0,'beta_revenue':0.0,'gamma_diversity':0.2,'delta_health':0.9},
}

for name, w in strategies.items():
    out = weighted_rerank(retr, emb_for(retr), w, final_k=5,
                          budget_cents=demo_filters['budget_cents'])
    artists = ' | '.join(out['artist'].str[:12].tolist())
    tiers   = ' | '.join(out['tier'].tolist())
    print('--- {:20s} ---'.format(name))
    print('  artists:', artists)
    print('  tiers:  ', tiers, '\n')


---
# Stage 6 — Diversity Strategies, Compared

Three approaches we should consider, with the prototype's round-robin as the baseline:

| Strategy | How | When it wins | When it loses |
|---|---|---|---|
| **Round-robin by `artist_id` (current)** | One slot per artist before second slot | Trivially explainable; fights single-artist domination | Doesn't see stylistic similarity across artists |
| **MMR (proposed γ)** | Greedy: max `α·rel - (1-α)·max_sim_to_shown` | Diversifies on *style*, not just identity | Greedy local optimum; harder to explain |
| **Hard quota** (`max_per_artist=1`) | Cap each artist | Like round-robin but doesn't allow rotation back | Can starve top result set if one artist truly dominates relevance |
| **Tier quota** | Reserve N slots for emerging | Direct lever for proposal §4's emerging-artist commitment | Can surface weak emerging matches if pool is shallow |

**Recommendation:** MMR for stylistic diversity (γ) **plus** a soft tier quota via δ — belt and braces. We'll measure that combination against the alternatives in Stage 8.

Below: hard-quota implementation for comparison.

In [ ]:
# NOTE — production v1's `applyQuota()` matches this function's
# behaviour: target = ceil(K × 0.2), swap lowest non-emerging for
# highest emerging that cleared QUOTA_FLOOR_REL. Aligned.

def quota_rerank(retrieved_df, max_per_artist=1, min_emerging_slots=2, final_k=FINAL_K):
    """Standalone quota-only ranker (no MMR, no health). For comparison purposes."""
    df = retrieved_df.sort_values('relevance', ascending=False).reset_index(drop=True)
    selected = []
    artist_counts = Counter()
    for _, r in df.iterrows():
        if sum(1 for s in selected if s['tier']=='emerging') >= min_emerging_slots:
            break
        if r['tier']=='emerging' and artist_counts[r['artist']] < max_per_artist:
            selected.append(r.to_dict()); artist_counts[r['artist']] += 1
    seen = {s['work_id'] for s in selected}
    for _, r in df.iterrows():
        if len(selected) >= final_k: break
        if r['work_id'] in seen: continue
        if artist_counts[r['artist']] >= max_per_artist: continue
        selected.append(r.to_dict()); artist_counts[r['artist']] += 1
    return pd.DataFrame(selected)


def apply_quota(ranked_df, retrieved_df, retrieved_emb_norm,
                final_k=FINAL_K, quota_floor_rel=QUOTA_FLOOR_REL):
    """POST-RERANK quota repair: ensure top-K contains at least ceil(K * 0.2)
    emerging-tier artists (1 of 5, 2 of 10, etc).

    Strategy: if not enough emerging artists in the current top-K cleared the
    relevance floor, swap the LOWEST-ranked non-emerging slot with the
    HIGHEST-ranked emerging artist OUTSIDE top-K who DID clear the floor.

    The quota is PERMISSIVE — if no emerging artist cleared the relevance floor,
    we DO NOT swap in a weak match just to fill the quota. Better to underfill
    than to surface garbage.

    Returns (repaired_df, swap_log) where swap_log records each swap.
    Swapped-in slots are marked with `quota_repaired=True`.
    """
    target_emerging = max(1, int(np.ceil(final_k * 0.2)))   # 1 of 5, 2 of 10
    df = ranked_df.copy().reset_index(drop=True)
    if 'quota_repaired' not in df.columns:
        df['quota_repaired'] = False

    current_emerging = (df['tier'] == 'emerging').sum()
    if current_emerging >= target_emerging:
        return df, []

    selected_ids = set(df['work_id'])
    pool = retrieved_df[
        (retrieved_df['tier'] == 'emerging')
        & (retrieved_df['relevance'] >= quota_floor_rel)
        & (~retrieved_df['work_id'].isin(selected_ids))
    ].sort_values('relevance', ascending=False)

    swap_log = []
    needed = target_emerging - int(current_emerging)
    non_em = df[df['tier'] != 'emerging'].sort_values('rank', ascending=False)
    non_em_idx = list(non_em.index)

    for _, em_row in pool.iterrows():
        if needed <= 0 or not non_em_idx:
            break
        slot_idx = non_em_idx.pop(0)
        rank = df.at[slot_idx, 'rank']
        replaced_id = df.at[slot_idx, 'work_id']
        # Copy the emerging row's data into the slot, preserving rank
        for col in retrieved_df.columns:
            if col in df.columns:
                df.at[slot_idx, col] = em_row[col]
        df.at[slot_idx, 'rank'] = rank
        df.at[slot_idx, 'quota_repaired'] = True
        # Reset the score-breakdown columns — the swapped-in row's scores
        # were computed for a different selection context.
        df.at[slot_idx, 'rel'] = float(em_row['relevance'])
        for c in ['div', 'health', 'price_pen', 'final_score']:
            if c in df.columns:
                df.at[slot_idx, c] = float('nan')
        swap_log.append({'replaced': replaced_id, 'with': em_row['work_id'], 'rank': int(rank)})
        needed -= 1

    return df, swap_log


# ─── Side-by-side: weighted ranker, with quota OFF vs ON ─────────────
shared_q = 'atmospheric content for a travel vlog'
shared_filt, _ = apply_hard_filters(catalog, medium=['music','video','illustration'])
shared_retr = semantic_retrieve(shared_q, shared_filt, emb_for(shared_filt))

# Without quota
mmr_only = weighted_rerank(shared_retr, emb_for(shared_retr), WEIGHTS, final_k=5)

# With quota repair
mmr_quota, swap_log = apply_quota(mmr_only, shared_retr, emb_for(shared_retr), final_k=5)

print('Query:', shared_q)
print('\n=== WEIGHTED RERANK ONLY (no quota) ===')
for _, r in mmr_only.iterrows():
    print('  {}. {:14s} ({:11s}) — {}'.format(
        int(r['rank']), r['artist'][:14], r['tier'], r['title']))
emerging_before = (mmr_only['tier']=='emerging').sum()
print('  → {} emerging in top-5'.format(emerging_before))

print('\n=== WEIGHTED + QUOTA REPAIR (target ≥ 1 emerging in top-5) ===')
for _, r in mmr_quota.iterrows():
    flag = ' [quota]' if r.get('quota_repaired') else ''
    print('  {}. {:14s} ({:11s}) — {}{}'.format(
        int(r['rank']), r['artist'][:14], r['tier'], r['title'], flag))
print('  → {} emerging in top-5'.format((mmr_quota['tier']=='emerging').sum()))
print('  → swap log:', swap_log if swap_log else '(no swap needed)')


---
# Stage 7 — Cold Start & Explore/Exploit

Session 4's most uncomfortable point: **the system's confidence is correlated with popularity, not necessarily quality.** New artists have:

- thinner profiles → noisier embeddings
- zero conversion data → β proxy is undefined
- no clicks/purchases → the self-reinforcing loop locks them out

Our prototype has **30 artists, 148 works**, with a long tail (Oscar Dias, Kenneth Eko: <500 reach, 1 work). Without explicit handling, those artists would never get exposure — and the platform fails its core promise.

## Simple operationalization for v1

**ε-greedy injection:** with probability ε ≈ 0.1, replace one slot in the top-K with a random emerging-artist work that cleared the relevance floor. Cheap, A/B-testable, easy to disable. The replaced slot is **tagged** so we can later measure whether explored items convert — that's how we close the loop.

## Future options

- **Thompson sampling** — sample each item's score from a posterior; high-uncertainty items get more chances
- **UCB** — score by mean + uncertainty bonus (deterministic Thompson)
- **Cold-start floor** — guarantee every new artist N impressions before being judged

In [ ]:
# NOTE — production v1 does NOT inject random emerging works.
# Deferred until catalog reaches ~1,000 works (see how_search_works.docx §6).
# With ~30 artists today, "random emerging" would be the same five people.

def epsilon_greedy_injection(ranked_df, candidate_pool_df, epsilon=0.1, min_rel=0.30):
    """With prob epsilon, replace the LAST slot with a random emerging-artist work
    that cleared min_rel and isn't already in the result set."""
    if random.random() > epsilon:
        return ranked_df, None
    eligible = candidate_pool_df[
        (candidate_pool_df['tier'] == 'emerging')
        & (candidate_pool_df['relevance'] >= min_rel)
        & (~candidate_pool_df['work_id'].isin(ranked_df['work_id']))
    ]
    if len(eligible) == 0:
        return ranked_df, None
    pick = eligible.sample(1).iloc[0]
    out = ranked_df.copy()
    last = out.index[-1]
    for c in pick.index:
        if c in out.columns:
            out.at[last, c] = pick[c]
    out.at[last, 'exploration_slot'] = True
    return out, pick['work_id']

# Demo: high ε to see it working
random.seed(0)
for trial in range(5):
    out, injected = epsilon_greedy_injection(proposed, retr, epsilon=0.5)
    print('Trial {}: injected={}'.format(trial+1, injected or '(no injection this trial)'))

---
# Stage 8 — Evaluation: Three Layers

Proposal §5 commits to three eval dimensions: **match quality, diversity check, failure mode audit**. Each gets a concrete metric below.

Eval is the part most teams skip and most regret skipping. Without it, *we cannot tell whether changes (different weights, different embedding model, different diversity strategy) make things better or worse*. The whole point of building this notebook is to make that measurement loop possible.

## Layer 1 — Offline IR metrics (component eval)

We need a **golden query set**: hand-labeled `(query, list-of-relevant-work-ids)` pairs. We start with 6 below; the production version should have 30–50.

From the labels we compute:

- **Recall@K** — of the relevant items, what fraction made it into the top-K?
- **Precision@K** — of the top-K, what fraction is relevant?
- **MRR** — `1/rank` of the first relevant result
- **NDCG@K** — discounted gain weighted by relevance grade

## Layer 2 — Marketplace metrics (system eval)

Run across many queries:

- **Distinct artists shown** — proxy for breadth
- **Emerging-artist share** in top-K — the proposal's diversity check, made concrete
- **Gini on artist-impression distribution** — Session 4's monoculture warning
- **Catalog coverage** — what % of works ever appear

## Layer 3 — Rubric-based judgment (semantic eval)

Session 7's framing: **prompt + world + rubric**. For each query, define a rubric of binary criteria; have humans (proposal §5: team rates ≥70% as plausibly relevant) or an LLM-as-judge score each top-K result. Catches things IR metrics miss — vibe matches, license suitability, tone.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Layer 1 — Golden set (in production: move to its own JSON or
# Airtable so non-technical teammates can add labels).
#
# Each item:
#   query    — natural-language buyer brief
#   filters  — implied hard requirements (uses `budget_cents` for soft pricing)
#   relevant — {work_id: grade}, grade in {1,2,3} (3 = perfect match)
#
# Note: work_ids match the REAL seed_data.json titles (so this golden set
# evaluates against the same catalog the prototype actually serves).
# ═══════════════════════════════════════════════════════════════

golden_set = [
    {'query':'lofi instrumental with rain samples for an evening vlog under $80',
     'filters':{'medium':'music','budget_cents':8000},
     'relevant':{'Noah Patel/Midnight loop':3,
                 'Kenji Arai/Whispers of Rain':3,
                 'Aya Tolentino/Rainy Day Reverie':2,
                 'Noah Patel/Window light':2}},

    {'query':'slow ambient music for a nature documentary, with field recordings',
     'filters':{'medium':'music'},
     'relevant':{'Kenji Arai/Dawn Chorus':3,
                 'Kenji Arai/Whispers of Rain':2,
                 'Sora Kim/Aurora drift':2}},

    {'query':'foggy coastal morning watercolor illustration',
     'filters':{'medium':'illustration'},
     'relevant':{'Ava Chen/Ocean fog':3}},

    {'query':'editorial illustration of mythical creatures in nature',
     'filters':{'medium':'illustration'},
     'relevant':{'Pilar Vega/Whispering Ferns':3,
                 'Pilar Vega/Sunlit Olive Grove':2}},

    {'query':'slow cinematic video footage of fog and water for travel content',
     'filters':{'medium':'video'},
     'relevant':{'Leo Brown/Quiet harbor':3,
                 'Leo Brown/Tidepool':3,
                 'Oscar Dias/Tides of Memory':2}},

    {'query':'fantasy character designs for an indie game',
     'filters':{'medium':'character'},
     'relevant':{'Maya Gomez/Fox warrior':3,
                 'Maya Gomez/Sky whale':2,
                 'Wren Hollis/Goblin Market':3,
                 'Asher Coen/Fabled Encounters':2}},
]

def evaluate_query(item, weights, k=FINAL_K, with_quota=True):
    """Run pipeline; compute per-query metrics. `with_quota` toggles post-rerank quota repair."""
    f = item['filters']
    filt, _ = apply_hard_filters(catalog, **f)
    if len(filt) == 0:
        return None
    retr = semantic_retrieve(item['query'], filt, emb_for(filt), top_k=RETRIEVAL_K)
    if len(retr) == 0:
        return None
    ranked = weighted_rerank(retr, emb_for(retr), weights, final_k=k,
                             budget_cents=f.get('budget_cents'))
    if with_quota:
        ranked, _swaps = apply_quota(ranked, retr, emb_for(retr), final_k=k)
    top_ids = ranked['work_id'].tolist()
    rel_ids = item['relevant']

    hits = [w for w in top_ids if w in rel_ids]
    precision = len(hits) / max(1, len(top_ids))
    recall    = len(hits) / max(1, len(rel_ids))
    mrr = next((1.0/r for r, w in enumerate(top_ids, 1) if w in rel_ids), 0.0)
    dcg = sum(rel_ids.get(w, 0) / math.log2(r + 1) for r, w in enumerate(top_ids, 1))
    ideal = sorted(rel_ids.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(r + 1) for r, g in enumerate(ideal, 1)) or 1.0
    return {'query': item['query'][:55] + '…',
            'precision@k': round(precision,3), 'recall@k': round(recall,3),
            'mrr': round(mrr,3), 'ndcg@k': round(dcg/idcg,3),
            'top_ids': top_ids, 'budget_cents': f.get('budget_cents')}

rows = [evaluate_query(it, WEIGHTS) for it in golden_set]
metrics_df = pd.DataFrame([r for r in rows if r])
print('=== Layer 1: Offline IR metrics with our_proposal weights (quota ON) ===\n')
print(metrics_df[['query','precision@k','recall@k','mrr','ndcg@k']].to_string(index=False))
print('\nMean: precision={:.3f}  recall={:.3f}  mrr={:.3f}  ndcg={:.3f}'.format(
    metrics_df['precision@k'].mean(), metrics_df['recall@k'].mean(),
    metrics_df['mrr'].mean(), metrics_df['ndcg@k'].mean()))


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Layer 2 — Marketplace metrics across the golden set, by strategy
#
# New v1 metrics (mission-critical):
#   - emerging_share_at_k  : % of impressions going to emerging-tier artists
#   - price_fit_pct        : % of results within original budget (soft band)
#   - quota_satisfaction   : did the quota's emerging target get met naturally?
# ═══════════════════════════════════════════════════════════════

def gini(values):
    v = sorted(values); n = len(v); s = sum(v)
    if n == 0 or s == 0: return 0.0
    return (2*sum((i+1)*x for i, x in enumerate(v)) - (n+1)*s) / (n*s)

def marketplace_metrics(weights, with_quota=True):
    artist_imps = Counter(); tier_imps = Counter(); seen = set()
    in_budget = 0; total_with_budget = 0
    quota_satisfied_natively = 0; quota_evaluated = 0
    target_emerging_per_q = max(1, int(np.ceil(FINAL_K * 0.2)))

    for it in golden_set:
        f = it['filters']
        # We need both the WITHOUT-quota and WITH-quota rankings to compute
        # natural-quota-satisfaction. Run both and use the requested one for impressions.
        m_no_quota = evaluate_query(it, weights, with_quota=False)
        m = evaluate_query(it, weights, with_quota=with_quota)
        if not m: continue

        # Natural quota satisfaction: did the unrepaired ranker hit the emerging target?
        if m_no_quota:
            quota_evaluated += 1
            unrep_top_ids = m_no_quota['top_ids']
            unrep_emerging = sum(
                1 for wid in unrep_top_ids
                if catalog.loc[catalog['work_id']==wid, 'tier'].iloc[0] == 'emerging'
            )
            if unrep_emerging >= target_emerging_per_q:
                quota_satisfied_natively += 1

        for wid in m['top_ids']:
            row = catalog[catalog['work_id']==wid].iloc[0]
            artist_imps[row['artist']] += 1
            tier_imps[row['tier']] += 1
            seen.add(wid)
            if f.get('budget_cents') is not None:
                total_with_budget += 1
                if row['price_from_cents'] <= f['budget_cents']:
                    in_budget += 1

    total = sum(artist_imps.values()) or 1
    return {
        'distinct_artists':         len(artist_imps),
        'distinct_works':           len(seen),
        'coverage':                 round(len(seen)/len(catalog),3),
        'emerging_share':           round(tier_imps['emerging']/total,3),
        'growing_share':            round(tier_imps['growing']/total,3),
        'established_share':        round(tier_imps['established']/total,3),
        'artist_gini':              round(gini(list(artist_imps.values())),3),
        'price_fit_pct':            round(in_budget/total_with_budget, 3) if total_with_budget else None,
        'quota_natural_rate':       round(quota_satisfied_natively/quota_evaluated, 3) if quota_evaluated else None,
    }

rows = []
for name, w in strategies.items():
    rows.append({'strategy': name, **marketplace_metrics(w)})
print('=== Layer 2: Marketplace metrics by strategy (quota ON) ===\n')
print(pd.DataFrame(rows).to_string(index=False))


### How to read this table

- **`pure_relevance`** is the strawman — pure cosine, no diversity. Expect: low distinct artists, low emerging share, high Gini.
- **`rel_plus_diversity`** adds γ. Should bump distinct artists; emerging share roughly unchanged (γ doesn't know about tiers).
- **`our_proposal`** adds δ. Should improve emerging share AND distinct artists.
- **`tier_lean`** is what happens if we crank δ. Watch: precision@K probably drops as we surface weaker emerging matches.

**This is exactly the artifact that justifies the weight choice in proposal §4.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Layer 3 — Rubric-based LLM-as-judge (Session 7 framing).
# COMMENTED OUT to avoid charging your account by default.
# Uncomment to run; ~$0.001 per result.
# ═══════════════════════════════════════════════════════════════

rubric_example = {
    'query': 'lofi instrumental for an evening vlog under $80',
    'criteria': [
        'The work is music, not visual or video.',
        'The mood is lo-fi or chill — not high-energy or orchestral.',
        'The price is at most $80.',
        'The description suggests it would suit a vlog context.',
    ],
}

# def llm_rubric_judge(work_row, rubric, model='gpt-4o-mini'):
#     prompt = (
#         'Buyer query: "{}"\n\nCandidate result:\n  {} ({}, ${}, {}-tier)\n  {}\n\n'
#         'For each criterion, respond YES or NO with a 1-sentence reason. JSON list.\n\n{}'
#     ).format(rubric['query'], work_row['title'], work_row['medium'],
#              work_row['price_from_cents']//100, work_row['tier'], work_row['description'],
#              '\n'.join('- ' + c for c in rubric['criteria']))
#     resp = openai_client.chat.completions.create(
#         model=model, messages=[{'role':'user','content':prompt}],
#         temperature=0, response_format={'type':'json_object'})
#     # ... parse and return %-criteria-met

print('Rubric for the demo query:')
for c in rubric_example['criteria']:
    print('  -', c)
print('\n(Uncomment llm_rubric_judge above to run on real results.)')

---
# Stage 9 — Failure Mode Audit

Proposal §5: *"probe with ambiguous and adversarial queries to see where semantic matching breaks down."* This is where we *try* to break our own system before users do.

## A taxonomy of failures (from Sessions 4 + 7)

1. **Negation** — embeddings encode topic, not logic. *"Not slow"* is near *"slow."*
2. **Ambiguity** — *"something for my brand video"* — what brand? what style?
3. **Off-topic / adversarial** — gibberish or attempt to break the system.
4. **Contradictory constraints** — *"music under $50 by a famous orchestral composer"* — empty result.
5. **Gaming** — sellers stuff descriptions with high-relevance keywords (Amazon failure mode).
6. **Zero true matches** — query is reasonable but nothing in the catalog fits; the system returns the next-most-similar-but-irrelevant item.

## What we want the system to do

| Failure | Desired behavior |
|---|---|
| Negation | Detect via simple regex or LLM-rewrite; either rewrite the query or warn |
| Ambiguity | Return a diversified set spanning the ambiguity, OR ask a clarifier |
| Adversarial | Refuse with clear message |
| Empty after filters | Don't silently drop a filter; tell the buyer what they'd need to relax |
| Gaming | Spot-check via LLM judge; flag mismatch between description and actual content |
| Zero match | Return *"no strong match"* + best-of-bad as a low-confidence suggestion |

In [ ]:
adversarial_queries = [
    ('negation',       'background music that is NOT slow or moody'),
    ('ambiguity',      'something for my video'),
    ('off_topic',      'a recipe for chocolate cake'),
    ('contradictory',  'music under $20 by a famous orchestral composer'),
    ('zero_match',     'photorealistic painting of Mars surface volcanoes'),
    ('all_over_budget','high-end cinematic orchestral composition with $5 budget'),
]

print('Adversarial probes through the FULL pipeline (filter → retrieve → rerank → quota):\n')
for label, q in adversarial_queries:
    # Run through search_agent to see how the system handles each
    # (search_agent is defined in Stage 10 — at this point we just inspect retrieval)
    retr = semantic_retrieve(q, catalog, EMB_NORM, top_k=3)
    print('--- {} ---  query: {!r}'.format(label, q))
    for _, r in retr.iterrows():
        print('  rel={:.2f} | {:14s} | {:6s} | {}'.format(
            r['relevance'], r['artist'][:14], r['medium'], r['title']))
    if retr['relevance'].max() < 0.30:
        print('  ⚠️  Best relevance < 0.30 — system should return NO_MATCH at the agent level.')
    print()

# Special case: the over-budget probe — show that the price ceiling fires
print('--- detailed: all_over_budget ---')
filt, drop_log = apply_hard_filters(catalog, budget_cents=500, medium='music')   # $5 budget → $7.50 ceiling
print('After hard filter (budget=$5, ceiling=$7.50): {} survive'.format(len(filt)))
print('All works dropped via price_ceiling: {}'.format(
    sum(1 for _, reason, _ in drop_log if reason == 'price_ceiling')))
print('→ search_agent should return EMPTY_AFTER_FILTER with a relax suggestion.')


### Reading the failure-mode output

Look at the relevance values. Two patterns to discuss in group:

1. **Negation:** *"NOT slow or moody"* still surfaces lo-fi. Embeddings encode topic, not logic. The fix lives upstream — pre-process the query through a lightweight LLM rewrite that detects negation patterns.

2. **Off-topic / zero-match:** all relevances are low. The fix is the **relevance floor** from Stage 5 + a confidence tier in Stage 10 — the system says *"no strong match"* instead of confidently returning the highest-of-bad.

**These are the failures we have today** (run the same query against `lib/search.ts` — same outcome). Stage 10 closes the loop with a confidence-tier output.

---
# Stage 10 — Agent Harness

Session 7's central message: **the winning agent is not the most capable one — it is the most trusted one.** Capability without harness is a demo, not a product.

Even though our prototype is *retrieval-only* (not multi-step agentic), the harness checklist still applies. Going through it forces us to make ownership and trust decisions explicit.

## Session 7 checklist, applied to us

| Harness element | Our v1 answer |
|---|---|
| **Tool access** | One tool: `search(query, filters) → ranked works`. Future: `message_artist`, `request_quote`, `purchase`, `escrow`. |
| **Memory & session state** | Per-buyer search history (so we can refine on follow-up queries). Session 6: *"user intent evolves across turns."* |
| **Permissions & delegation** | Buyer permissions: `browse-only` vs. `can-purchase` vs. `can-commission` (different liability). Artist permissions: who can list, attestation tier unlocks pricing tier. |
| **Routing & retries** | Embedding API fails → fall back to Postgres `ts_rank` keyword search (degraded mode, log). |
| **Logs & audit** | Every result row carries the score breakdown (already implemented in Stage 5). Plus: query, filters, exploration-slot tag, returned-ids, ts. |
| **Payment handoff** | Out of scope for v1; future: Stripe + escrow until artist confirms delivery. |
| **Identity & attestation** | `attestation_tier` already in schema. Verified unlocks higher-tier listing (`seed.py` already prices verified at 1.8x). |
| **Deployment threshold** | We're at **Level 1 — Suggest only.** System surfaces matches; buyer decides and acts. We're not at Level 2 (act-with-confirmation) until a real purchase flow exists. |

## Failure modes the harness catches that the algorithm can't

- **Silent failure with high confidence** — algorithm says rel=0.42, UI shows "Recommended." Fix: **confidence tier** (HIGH / MEDIUM / LOW / NO_MATCH) gates the UI label.
- **Empty after filters** — algorithm returns 0 results, UI shows blank page. Fix: surface *"no works match — try relaxing one of these filters: …"*.
- **Wrong escalation** — buyer asks something the system can't answer ("can you commission this in 2 weeks?") and the system invents an answer instead of routing to the artist.

Below: `search_agent()` wraps the algorithm with audit logging, confidence tiering, and the failure-mode gates.

In [ ]:
# NOTE — production v1's `rankWorks()` / `rankArtists()` return the
# result array directly. The audit log + status codes here
# (OK / EMPTY_AFTER_FILTER / NO_MATCH) are deferred —
# see how_search_works.docx §6, "Structured request-level audit logging".

RELEVANCE_FLOORS = {'HIGH': 0.55, 'MEDIUM': 0.40, 'LOW': 0.25}   # below LOW → NO_MATCH

def confidence_tier(rel):
    if rel is None or (isinstance(rel, float) and np.isnan(rel)):
        return 'UNKNOWN'
    if rel >= RELEVANCE_FLOORS['HIGH']:    return 'HIGH'
    if rel >= RELEVANCE_FLOORS['MEDIUM']:  return 'MEDIUM'
    if rel >= RELEVANCE_FLOORS['LOW']:     return 'LOW'
    return 'NO_MATCH'

def search_agent(query, *, filters=None, weights=WEIGHTS, final_k=FINAL_K, log=None):
    """End-to-end search:
       hard filter → retrieve → weighted rerank → quota repair → confidence tier.

    Returns audit-logged result with status ∈ {OK, EMPTY_AFTER_FILTER, NO_MATCH}.
    """
    if log is None: log = []
    log.append(('query_received', {'query': query, 'filters': filters}))
    filters = filters or {}

    # Stage 1 — hard filters
    filt, drop_log = apply_hard_filters(catalog, **filters)
    log.append(('hard_filter', {'survivors': len(filt), 'dropped': len(drop_log)}))

    if len(filt) == 0:
        log.append(('exit', 'EMPTY_AFTER_FILTER'))
        return {'status': 'EMPTY_AFTER_FILTER', 'results': [], 'log': log,
                'message': 'No works match the hard requirements. Try relaxing one of: '
                           + ', '.join(filters.keys())}

    # Stage 2 — retrieve
    retr = semantic_retrieve(query, filt, emb_for(filt), top_k=RETRIEVAL_K)
    top_rel = float(retr['relevance'].max()) if len(retr) else 0.0
    log.append(('semantic_retrieve', {'top_relevance': round(top_rel,3), 'returned': len(retr)}))

    if top_rel < RELEVANCE_FLOORS['LOW']:
        log.append(('exit', 'NO_MATCH'))
        return {'status': 'NO_MATCH', 'results': [], 'log': log,
                'message': 'No strong match. Best relevance: {:.2f} (floor: {})'.format(
                    top_rel, RELEVANCE_FLOORS['LOW'])}

    # Stage 3 — weighted rerank
    ranked = weighted_rerank(retr, emb_for(retr), weights, final_k=final_k,
                             budget_cents=filters.get('budget_cents'))

    # Stage 4 — quota repair
    repaired, swap_log = apply_quota(ranked, retr, emb_for(retr), final_k=final_k)
    if swap_log:
        log.append(('quota_repair', {'swaps': swap_log}))
    repaired['confidence'] = repaired['rel'].apply(confidence_tier)
    log.append(('rerank', {'final_k': len(repaired),
                           'confidence_mix': dict(Counter(repaired['confidence']))}))

    return {'status': 'OK', 'results': repaired.to_dict('records'),
            'log': log, 'deployment_level': 'SUGGEST_ONLY'}

# Demo — full agent run with audit log
result = search_agent('moody lofi for an evening vlog',
                     filters={'medium':'music','budget_cents':6500})
print('Status:', result['status'])
print('Deployment level:', result.get('deployment_level'))
print('\n=== Results ===')
for r in result['results']:
    qflag = ' [quota]' if r.get('quota_repaired') else ''
    over = ' *' if r.get('price_from_cents', 0) > 6500 else '  '
    print('  [{:8s}] {:14s} ({:11s}) | ${:>3}{} | rel={:.2f}{}  | {}'.format(
        r['confidence'], r['artist'][:14], r['tier'],
        r['price_from_cents']//100, over,
        r['rel'] if not (isinstance(r['rel'], float) and np.isnan(r['rel'])) else 0.0,
        qflag, r['title']))
print('   (* = within soft band, over budget)')
print('\n=== Audit log ===')
for step, info in result['log']:
    print('  {:20s} {}'.format(step, info))


---
# The Replicable Procedure

If we strip away the artist-marketplace specifics, this is the **template** for any future search/matching project. Print this page and tape it to the wall.

## The 10-step procedure

1. **Measure the baseline.** Whatever ranker you have today, run it on real data. Don't redesign in the dark.
2. **Frame the objective.** Write the weighted formula `α·X + β·Y + γ·Z + …`. **The weights are the strategy.**
3. **Define operational tiers.** Turn fuzzy concepts ("emerging artist") into checkable conditions on real columns.
4. **Move hard requirements out of scoring.** Constraints go in SQL `WHERE`, not in embedded text or client-side `.filter()`.
5. **Retrieve before you rerank.** Top-K candidates from cheap, fast similarity. Reranking is for the survivors.
6. **Rerank with the full objective.** Log every component score so you can audit. Use a relevance floor.
7. **Pick a diversity strategy.** MMR, quota, or both. Justify with marketplace metrics.
8. **Address cold start explicitly.** ε-greedy injection (or Thompson, UCB). Tag exploration slots.
9. **Build the 3-layer eval before you ship.** Offline IR + marketplace + rubric. Without it, you can't tell if changes help.
10. **Wrap in an agent harness.** Tool access, memory, permissions, logs, deployment threshold.

## Status of our prototype against the procedure

| Step | Today (`lib/search.ts`) | After this notebook ships |
|---|---|---|
| 1. Baseline measured | ❌ no eval | ✅ Stage 0 + Stage 8 |
| 2. Objective | ❌ implicit (α only) | ✅ explicit `WEIGHTS` config |
| 3. Operational tier | ❌ none | ✅ `tier_of(reach_score)` |
| 4. Hard filters | ❌ client-side | ✅ SQL filter (migration) |
| 5. Retrieve | ✅ HNSW + cosine | ✅ unchanged |
| 6. Rerank with breakdown | ❌ round-robin only, no breakdown | ✅ `weighted_rerank` |
| 7. Diversity strategy | ⚠️ round-robin (artist only) | ✅ MMR + δ tier boost |
| 8. Cold start | ❌ none | ✅ ε-greedy injection |
| 9. Eval | ❌ none | ✅ golden set + 3 layers |
| 10. Agent harness | ❌ raw results, no tiers | ✅ `search_agent` wrapper |

---
# Discussion Questions (For the Group Meeting)

## On objective design

1. Are the proposed v1 weights `α=1.0, β=0.0, γ=0.4, δ=0.3` reasonable? Where do they go in v2?
2. Is δ really separate from γ, or is δ just "diversity along the tier axis"?
3. What's our trigger to turn β on? Number of artists? Number of completed transactions?
4. Should `attestation_tier=verified` feed a fifth term (trust)? Or is it best left as a hard filter?

## On the operational tier definition

5. We picked `reach_score < 1500 = emerging`. Is that the right threshold for our seed (~30 artists)? At scale (10k artists), the threshold might be very different.
6. Should the tier be quantile-based ("bottom 30% of reach") rather than absolute?

## On hard filters vs. soft preferences

7. Right now refine chips like `warmer`, `darker` are appended to the embedding query. Is that the right call (treat them as soft preferences) or do we need more structure?
8. Is price always a hard filter, or should the buyer toggle "strict" / "flexible"?

## On diversity

9. MMR (γ) **plus** quota (via δ) — belt and braces, or pick one?
10. How do we prevent diversity from being gamed? An artist could spread descriptions across stylistic clusters to occupy more slots.

## On evaluation

11. Who labels the golden set? Should we recruit 2–3 buyer personas (KOL, AI training data buyer, indie filmmaker) to validate?
12. Proposal §5 targets *"≥70% rated at least plausibly relevant"*. Is that Precision@5 ≥ 0.70, or % of queries where Recall@5 ≥ 1?
13. When do we shift from internal eval to live A/B on the prototype? Session 4 warned standard A/B breaks on platforms (switchback experiments).

## On agent harness

14. We're at Level 1 (Suggest only). What unlocks Level 2 (Act with confirmation)? Purchase flow? Escrow? Identity verification?
15. Does the UI need to show the score breakdown (transparency) or just the confidence tier (simplicity)?

## On Stage 9 failures

16. Of the failures (negation, ambiguity, off-topic, contradictory, zero-match), which are *acceptable* vs. *must-fix*?
17. Do we need an LLM-rewrite layer in front of the embedder for negation/ambiguity?

## Stretch

18. Thompson sampling vs. ε-greedy — is the complexity worth it at our scale?
19. Should we fine-tune our embeddings on user clicks (Airbnb pattern from Session 4) once we have interaction data?

---
# Appendix — How to Ship This to the Live Prototype

Three concrete change-points to get from this notebook into production code.

## A. SQL migration — push hard filters into `search_works`

New file `supabase/migrations/0003_search_works_with_filters.sql`:

```sql
-- Replace search_works with a filter-aware variant.
create or replace function public.search_works(
  query_embedding   vector(1536),
  match_count       int default 50,
  filter_medium     medium_kind default null,
  filter_price_max  int default null,
  filter_verified   boolean default false
) returns table (
  work_id           uuid,
  artist_id         uuid,
  title             text,
  medium            medium_kind,
  price_from_cents  int,
  attestation_tier  attestation_tier,
  reach_score       int,
  similarity        float
)
language sql stable as $$
  select
    w.id, w.artist_id, w.title, w.medium, w.price_from_cents,
    a.attestation_tier, a.reach_score,
    1 - (w.embedding <=> query_embedding) as similarity
  from public.works w
  join public.artists a on a.id = w.artist_id
  where w.embedding is not null
    and (filter_medium    is null or w.medium = filter_medium)
    and (filter_price_max is null or w.price_from_cents <= filter_price_max)
    and (not filter_verified or a.attestation_tier = 'verified')
  order by w.embedding <=> query_embedding
  limit match_count;
$$;
```

## B. TypeScript — `web/lib/search.ts` rewrite

Replace `rankWorks` with a `weightedRerank` that mirrors the Python in this notebook. Skeleton:

```typescript
export type SearchFilters = {
  medium?: 'music'|'illustration'|'video'|'character';
  priceMaxCents?: number;
  verifiedOnly?: boolean;
};

export type RankWeights = {
  alphaRelevance: number;  // 1.0
  betaRevenue:    number;  // 0.0
  gammaDiversity: number;  // 0.4
  deltaHealth:    number;  // 0.3
};

export type WorkResult = {
  artist_id: string;
  work_id: string;
  title: string;
  medium: string;
  rel: number;     // breakdown
  div: number;
  health: number;
  score: number;   // final
  confidence: 'HIGH'|'MEDIUM'|'LOW'|'NO_MATCH';
};

// Pseudocode:
//   const candidates = await searchWorksRpc(vec, k * 5, filters);
//   return weightedRerank(candidates, candidateEmbeddings, weights, k);
```

**Important:** the RPC currently doesn't return embeddings, so MMR can't be computed in TypeScript without re-fetching. Two options:

1. **Have the RPC return embeddings** in the candidate set (1536 floats × 50 = manageable payload, ~300KB)
2. **Compute MMR in Postgres** via a custom function — cleaner but more SQL

Option 1 is simpler for v1.

## C. Frontend — send structured filters

`SearchPage.tsx`: stop appending refine chips that are constraints to the embedded query string. Send them as structured filters:

```typescript
const body = {
  query: q,                              // free text only — vibes go here
  filters: {                             // hard filters → SQL
    medium: medium === 'any' ? undefined : medium,
    priceMaxCents: applied.has('under $200') ? 20000
                  : applied.has('$200–500') ? 50000 : undefined,
    verifiedOnly,
  },
  weights: { ... },                      // optional override; default = our_proposal
  limit: 20,
};
```

## D. Eval harness — `scripts/eval.ts`

A thin script that loads `eval/golden_set.json`, calls the live `/api/search` for each, and prints the same Layer-1 + Layer-2 metrics this notebook produces. Run it manually before merging any ranker change. Add to CI eventually.

## E. Telemetry — audit log

Wherever `search_agent` lives in production, append every call to a Supabase `search_logs` table:

```sql
create table if not exists public.search_logs (
  id          uuid primary key default gen_random_uuid(),
  user_id     uuid,
  query       text,
  filters     jsonb,
  result_ids  uuid[],
  exploration_slot_id uuid,
  ts          timestamptz default now()
);
```

Without this, **we can't compute exploration-slot conversion rates later** — meaning the cold-start loop never closes. This is the single most important persistence to add early.

---

**End of notebook.** Bring the **Discussion Questions** section + this Appendix to the next group meeting.